# Programación Paralela en Python


### Por qué programación paralela?

Muchas tareas se pueden ejecutar de manera simultanea y todos los dispositivos que usamos diariamente ahora estan compuestos de multiples procesadores. 

Todo el software que diseñamos e implementamos tiene que tomar en cuenta esta disposicion para podes optimizar y aprovechar los recursos. 

Todo software moderno utiliza esto en su codigo para aplicaciones como:
- Procesamiento de imágenes
- Procesamiento, edicion y maipulacion de video.
- Simulaciones científicas.
- Servidores web que atienden múltiples usuarios.

Esta es una introduccion con Python 

Formas simples de trabajar en paralelo para aprovechar mejor los múltiples núcleos del CPU.


# Cuantos nucleos tengo en mi dispositivo?

In [7]:
import multiprocessing

multiprocessing.cpu_count()

8

# Paso 1

primero tenemos una función

In [8]:
import time

def tarea_lenta(n):
    print(f"Iniciando tarea {n}")
    time.sleep(2)
    print(f"Tarea {n} terminada")
    return n


# Paso 2

vamos a ejecutar esta funcion de forma secuencial

In [9]:

print("secuencial")
inicio = time.time()
for i in range(4):
    tarea_lenta(i)
fin = time.time()
print(f"Tiempo total: {fin - inicio:.2f} segundos")

secuencial
Iniciando tarea 0
Tarea 0 terminada
Iniciando tarea 1
Tarea 1 terminada
Iniciando tarea 2
Tarea 2 terminada
Iniciando tarea 3
Tarea 3 terminada
Tiempo total: 8.02 segundos




# Que ocurrió en la ejecución ?


Cada una de las tareas se ejecutaron una despues de otra (Forma secuencial). Esto se evidencia desde el tiempo de ejecución, cada tarea dura aproximadamente 2s, al ser 4 tareas podemos ver el tiempo siendo 8s. 

# Paso 3

Ahora vamos a implementar ejecucion paralela para realizar la tarea la misma cantidad de veces pero en *Paralelo*

In [10]:
from multiprocessing import Process

procesos = []
inicio = time.time()

for i in range(4):
    p = Process(target=tarea_lenta, args=(i,))
    procesos.append(p)
    p.start()

for p in procesos:
    p.join()

fin = time.time()
print(f"Tiempo total: {fin - inicio:.2f} segundos")


Tiempo total: 0.15 segundos


Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=89, pipe_handle=96)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'tarea_lenta'
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=89, pipe_handle=99)
                                              

pudimos correrla aqui?

No se pudo correr. Era necesario levantar otro thread como se ve en el archivo paso3.py al llamar al metodo main. Algo que pudimos observar es que el sistemas operativos Windows se ve como si "ejecuta" el programa porque muestra un tiempo de ejecución pero no muestra ningun print de ejecución de las tareas. 

*abrir el arvchivo **paso3.py***

# Paso 4

1. **¿Qué hay diferente en el código anterior al código que está en el archivo?**

   El código del notebook **no tiene** la guarda `if __name__ == "__main__":`, mientras que `paso3.py` **sí la tiene**. Todo el bloque que crea y lanza los procesos está dentro de esa condición en el archivo.

2. **¿Qué significa esa diferencia?**

   En macOS y Windows, `multiprocessing` usa el método **spawn** para crear procesos hijos: cada proceso hijo reimporta el módulo `__main__` desde cero. Sin la guarda `if __name__ == "__main__":`, ese código de creación de procesos se ejecutaría también en cada proceso hijo, causando el error `AttributeError: module '__main__' has no attribute 'tarea_lenta'` que vimos (el notebook no es un módulo importable normal). Con la guarda, el bloque solo corre cuando el script es el punto de entrada directo, **no** cuando es reimportado por un proceso hijo.

3. **¿Qué ocurrió con la ejecución anterior de forma diferente?**

   En el notebook (sin guarda), los procesos hijos intentaron reimportar `__main__` y no encontraron `tarea_lenta`, fallando con `AttributeError`. El tiempo que mostró (0.09s) solo mide el lanzamiento de los procesos, no su ejecución real — nunca corrieron. En `paso3.py` (con guarda), los procesos hijos cargan correctamente `tarea_lenta` y el programa funciona en paralelo como se espera.

# Paso 5

Ahora utilizaremos un Process Pool

In [11]:
from concurrent.futures import ProcessPoolExecutor

inicio = time.time()

with ProcessPoolExecutor() as executor:
    resultados = list(executor.map(tarea_lenta, range(4)))

print("Resultados:", resultados)
fin = time.time()
print(f"Tiempo total: {fin - inicio:.2f} segundos")

Process SpawnProcess-19:
Process SpawnProcess-20:
Process SpawnProcess-17:
Process SpawnProcess-18:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.


# Ejercicio

1. Crear una función que calcule si un número es primo.

2. Genera una lista de 20 números grandes y verifica cuántos son primos.

3. Hacer la ejecucion secuencialmente y mostrar los tiempos.

4. Hacer con concurrent.futures y mostrar tiempos.


In [13]:
import random
import time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

def es_primo(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
    return True

random.seed(42)
numeros_grandes = [random.randint(10**12, 10**13) for _ in range(20)]

# --- Secuencial ---
inicio = time.time()
resultados_seq = [es_primo(n) for n in numeros_grandes]
fin = time.time()
primos_seq = sum(resultados_seq)
print(f"Secuencial  → {primos_seq} primos | {fin - inicio:.3f} s")

# --- Paralelo (fork: hereda la memoria del padre, no necesita reimportar __main__) ---
inicio = time.time()
ctx = multiprocessing.get_context('fork')
with ProcessPoolExecutor(mp_context=ctx) as executor:
    resultados_par = list(executor.map(es_primo, numeros_grandes))
fin = time.time()
primos_par = sum(resultados_par)
print(f"Paralelo    → {primos_par} primos | {fin - inicio:.3f} s")

Secuencial  → 2 primos | 0.044 s
Paralelo    → 2 primos | 0.097 s


Es mas rapido por el tipo de tarea. Es mas costoso crear nuevos proceso que trabajarlo de forma secuencial 